# 02. Baseline — LightGBM / XGBoost / CatBoost

特徴量エンジニアリングをせず、デフォルトパラメータで基準となるスコアを作る。

ここで EDA の **施策1「カテゴリ列を落とさずモデルに渡す」** を検証する。

1. **数値列のみ**(7列)で学習する
2. **数値列 + カテゴリ列**(13列)で学習する。カテゴリはエンコードせず、各ライブラリのネイティブなカテゴリ対応に渡す

CV は全モデル共通で `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`。
この分割を全モデルで揃えることで、あとから OOF 同士をアンサンブルできる。

同じ処理は `src/02_baseline_<model>.py` でも実行でき、2つのスコアを print する。

In [1]:
import os, sys
# リポジトリルートを作業ディレクトリにして、data/ などの相対パスを揃える
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
# 絶対パスは出力しない(公開リポジトリに個人のディレクトリ構成を残さないため)
print("data/ を検出:", os.path.isdir("data"))

data/ を検出: True


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

TARGET = "Will_Buy_EV"
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

NUM_COLS = [c for c in train.select_dtypes(include=[np.number]).columns if c != "id"]
CAT_COLS = [c for c in train.columns if c not in NUM_COLS + ["id", TARGET]]
y = (train[TARGET] == "Yes").astype(int)

# LightGBM / XGBoost 用: train/test で共通のカテゴリ集合を持つ category 型にする
X, X_test = train[NUM_COLS + CAT_COLS].copy(), test[NUM_COLS + CAT_COLS].copy()
for c in CAT_COLS:
    cats = pd.concat([train[c], test[c]]).astype("category").cat.categories
    X[c] = pd.Categorical(train[c], categories=cats)
    X_test[c] = pd.Categorical(test[c], categories=cats)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("数値列:", NUM_COLS)
print("カテゴリ列:", CAT_COLS)

数値列: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
カテゴリ列: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


## 共通の CV ループ

fold ごとに学習して OOF 予測を作り、OOF AUC を返す。
`cols` で使う列を切り替えて、数値列のみ / 数値列 + カテゴリ列を同じ条件で比べる。

In [3]:
def run_cv(make_model, X, cols, fit_kwargs=None):
    oof = np.zeros(len(X))
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        model = make_model()
        model.fit(X[cols].iloc[tr], y.iloc[tr], **(fit_kwargs or {}))
        oof[va] = model.predict_proba(X[cols].iloc[va])[:, 1]
    auc = roc_auc_score(y, oof)
    print(f"  {len(cols)}列  OOF AUC: {auc:.5f}")
    return auc


results = {}

## LightGBM — `category` dtype をそのまま渡す

In [4]:
from lightgbm import LGBMClassifier

make = lambda: LGBMClassifier(random_state=42, verbosity=-1)
results["LightGBM"] = (run_cv(make, X, NUM_COLS), run_cv(make, X, NUM_COLS + CAT_COLS))

  7列  OOF AUC: 0.88380


  13列  OOF AUC: 0.94123


## XGBoost — `enable_categorical=True`

In [5]:
from xgboost import XGBClassifier

make = lambda: XGBClassifier(random_state=42, tree_method="hist", enable_categorical=True)
results["XGBoost"] = (run_cv(make, X, NUM_COLS), run_cv(make, X, NUM_COLS + CAT_COLS))

  7列  OOF AUC: 0.88362


  13列  OOF AUC: 0.94124


## CatBoost — `cat_features` に列名を渡す

CatBoost はカテゴリを文字列のまま受け取る。**所要 30 分程度**。

In [6]:
from catboost import CatBoostClassifier

Xc = train[NUM_COLS + CAT_COLS].copy()
make = lambda: CatBoostClassifier(random_state=42, verbose=False, allow_writing_files=False)
results["CatBoost"] = (run_cv(make, Xc, NUM_COLS),
                       run_cv(make, Xc, NUM_COLS + CAT_COLS, fit_kwargs={"cat_features": CAT_COLS}))

  7列  OOF AUC: 0.88398


  13列  OOF AUC: 0.94156


In [7]:
pd.DataFrame(
    [(m, a, b, b - a) for m, (a, b) in results.items()],
    columns=["モデル", "数値列のみ", "数値列 + カテゴリ列", "差"],
).round(5)

,モデル,数値列のみ,数値列 + カテゴリ列,差
0,LightGBM,0.88380,0.94123,0.05743
1,XGBoost,0.88362,0.94124,0.05761
2,CatBoost,0.88398,0.94156,0.05758


## 結果 — 施策1 の検証

| モデル | OOF AUC<br>数値列のみ(7列) | OOF AUC<br>数値列 + カテゴリ列(13列) | 差 | Public LB<br>数値列 + カテゴリ列 |
|---|---|---|---|---|
| LightGBM | 0.88380 | **0.94123** | **+0.05743** | 0.94093 |
| XGBoost | 0.88362 | **0.94124** | **+0.05761** | 0.94152 |
| CatBoost | 0.88398 | **0.94156** | **+0.05758** | 0.94170 |

- **OOF AUC**: 学習データを 5 つに分けた交差検証のスコア(上のセルの実行結果)
- **Public LB**: Kaggle に提出し、テストデータの一部で採点されたスコア

対象のデータが違うので値は一致しない。差はどれも 0.0003 以内で、
交差検証のスコアが提出結果をおおむね再現できている(過学習していない)ことの確認にもなる。

**カテゴリ列を加えるだけで、3モデルとも OOF AUC が約 0.057 上がった。**
EDA で見たとおり、`Subsidy_Available`(購入率の幅 26.9 pt)と `Range_Anxiety_Level`(18.8 pt)は、
数値列の `Age`(4.1 pt)などよりはるかに強い。カテゴリ列を落とすと、この情報をまるごと失う。

以降の工程では、数値列 + カテゴリ列の13列を出発点にする。

次は EDA の施策2(値を丸めずに購入率を渡す)を検証する(→ `03_feature_engineering.ipynb`)。